In [115]:
!pip install langchain langchain-google-genai langchain-community langchain-core pypdf faiss-cpu

In [116]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import getpass

In [117]:
os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your gemini api key")

Enter your gemini api key··········


In [118]:
LLM = ChatGoogleGenerativeAI(
    model = 'gemini-3.6-flash'
)

In [119]:
from langchain_community.document_loaders import PyPDFLoader

In [120]:
LoadedContent = PyPDFLoader("/content/troubleshooting_runbook.pdf")

In [121]:
#Load the pdf document
Document = LoadedContent.load()

#Print number of pages
print("Total number of pages", len(Document))

Total number of pages 7


In [122]:
#Extract text from all pages
text = ""

for page in Document:
  text = page.page_content


print(text)


Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 7
Appendix: Quick Reference and Post-Incident
Checklist
A. Error code quick reference
Code
Meaning
First thing to check
Typical owner
ERR-401
Expired / invalid token
Token TTL config, NTP clock skew, JWKS cache
Team Sentinel
ERR-429
Rate limit exceeded
Top API keys, retry storms
Team Orbit / Sentinel
ERR-500
Unhandled exception
Recent deploy, top stack frame
Owning service team
ERR-503
Dependency unavailable
Vendor status, circuit breaker
Owning service team
ERR-504
Gateway timeout
Slow queries, missing client timeouts
Team Phoenix / Atlas
DB-1205
Lock wait timeout
pg_blocking_pids, long transactions
Team Atlas / Phoenix
DB-POOL
Pool exhausted
Connection leaks, pool x replicas
Team Atlas / Sentinel
MEM-OOM
OOMKilled (exit 137)
Memory limit vs working set, heap dump
Team Compass / Ledger
DISK-FULL
Disk above 95%
Logs, temp files, index shards
Team Atlas / Ledger
CERT-EXP
TLS cert expired
op

In [123]:
# Check for common text quality issues
issues = []

# Empty pages
empty_pages = [
    i + 1
    for i, page in enumerate(Document)
    if not page.page_content.strip()
]

if empty_pages:
    issues.append(f"Empty pages found: {empty_pages}")

# Excessive whitespace
if "  " in text:
    issues.append("Excessive whitespace detected")

# Broken words (caused by line wrapping)
if "-\n" in text:
    issues.append("Possible broken words detected")

print("Text Quality Issues:")
for issue in issues:
    print("-", issue)

Text Quality Issues:


In [124]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

#split text into chunks
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks=text_splitter.split_documents(Document)

In [125]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2'
)

In [126]:
from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)


In [127]:
question = "What are the golden rules?"
results = vector_store.similarity_search_with_score(query=question)

for i, (doc,score) in enumerate(results):
  print(f"RESULT{i}")
  print(f"SCORE{score}")
  print(f"Document{doc.page_content[:500]}")

RESULT0
SCORE0.6352295279502869
DocumentData
Team Ledger
Data warehouse
Nightly ETL
user-profile-api
Identity
Team Sentinel
PostgreSQL (profiles-db)
auth-service
4. Golden rules
1) Check recent deployments first: roughly one in four incidents in the log is deploy related, and a rollback
is almost always faster than a forward fix. 2) Never restart a stateful component before capturing logs and
a heap or thread dump. 3) Communicate early: post in the incident channel within five minutes of
acknowledging any Sev1 or Sev2. 4) Change one t
RESULT1
SCORE0.6397215127944946
Documentdiagnostic checklist in order, and apply the fix only after the cause is confirmed. If two consecutive
diagnostic steps produce no signal, escalate according to the severity matrix in Section 3. Always record
the root cause category in the incident ticket; reporting depends on it.
3. Service map
Service
Domain
Owning team
Primary datastore
Critical dependency
payment-gateway
Payments
Team Phoenix
PostgreSQL (payment

In [128]:
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs={"k":1}
)

In [129]:
from langchain_core.prompts import ChatPromptTemplate

In [130]:
rag_prompt = ChatPromptTemplate.from_template(
    """
You are a senior site reliability engineer and your job is to write an on-call runbook section based on the retrieved context.

Use only the retrieved context to write the runbook. Do not invent error codes, services, log counts, or fixes that are not in the context.

If a required detail is not available or cannot be found in the context, say - "I could not find this info" - and do not guess.

Retrieved context: {context}

Task: {question}

Write a runbook titled "Production Incident Runbook: Top 10 Error Codes" with the following:

1. Purpose and scope
2. Error code entries (one for each of the ten error codes)

Write in clear, imperative language so an engineer at 3 a.m. with no prior context can follow each step. Use headings for each section and numbered steps for checklists and fixes.

Provide:
1. Answer: the complete runbook
2. Supporting evidence: for each error code, cite the specific part of the retrieved context (log entry, ticket, or incident record) that supports the symptom, cause, and fix
"""
)

In [131]:
from langchain_core.output_parsers import StrOutputParser

In [132]:
rag_chain = rag_prompt|LLM|StrOutputParser()

In [133]:
question ="What are the golden rules?"

retrieved_docs=retriever.invoke(question)

In [134]:
context="\n\n".join([doc.page_content for doc in retrieved_docs])


In [135]:
response=rag_chain.invoke({"context":context, "question":question, })

In [136]:
print(response)

# Production Incident Runbook: Top 10 Error Codes

## 1. Purpose and Scope
This runbook guides on-call engineers responding to production incidents for systems including Team Ledger's Data warehouse and Nightly ETL, as well as Identity / Team Sentinel's `user-profile-api`, `auth-service`, and PostgreSQL (`profiles-db`).

### Golden Rules
1. **Check recent deployments first:** Roughly one in four incidents in the log is deploy-related, and a rollback is almost always faster than a forward fix.
2. **Preserve stateful logs/dumps:** Never restart a stateful component before capturing logs and a heap or thread dump.
3. **Communicate early:** Post in the incident channel within five minutes of acknowledging any Sev1 or Sev2 incident.
4. **Isolate changes:** Change one thing at a time and note the time of each change in the ticket.

---

## 2. Error Code Entries
I could not find this info.

*(Note: The retrieved context does not contain any error codes, symptoms, root causes, or specific step

1.**Description of the Embedding Model Used**


Embedding Model: gemini-embedding-2

The embedding model converts textual content into high-dimensional numerical vectors while preserving the semantic meaning of the text. Instead of comparing exact keywords, the model represents text based on its contextual meaning, enabling semantic search.

Why it is used in RAG:

Transforms document chunks into vector representations.
Enables similarity-based retrieval.
Helps find relevant content even when the query wording differs from the document wording.




2.**Explanation of Vector Storage Approach**

Vector Database Used: FAISS

After generating embeddings for each chunk, the vectors are stored in a FAISS vector index.


Storage Process

-PDF text is extracted.

-Text is split into chunks.

-Each chunk is converted into an embedding vector.

-Vectors are stored in a FAISS DB.

-User queries are also converted into embeddings.

-Similarity search retrieves the most relevant chunks.

In [137]:
#Implement query based retrieval

query = input("Enter your query: ")

k_values = [3, 5, 7]

for k in k_values:

    print(f"\nTop {k} Retrieved Chunks")

    docs = vector_store.similarity_search(
        query,
        k=k
    )

    for idx, doc in enumerate(docs, 1):

        print(f"\nChunk {idx}")
        print(doc.page_content[:300])

Enter your query: What should we do for the standard first five minitues?

Top 3 Retrieved Chunks

Chunk 1
debug; they coordinate, keep the timeline, and manage stakeholder communication. If the primary on-call
has not acknowledged an alert within the ack target, the page automatically escalates to the secondary
on-call and then to the team lead.
7. Standard first five minutes
 1. Acknowledge the alert a

Chunk 2
diagnostic checklist in order, and apply the fix only after the cause is confirmed. If two consecutive
diagnostic steps produce no signal, escalate according to the severity matrix in Section 3. Always record
the root cause category in the incident ticket; reporting depends on it.
3. Service map
Ser

Chunk 3
Data
Team Ledger
Data warehouse
Nightly ETL
user-profile-api
Identity
Team Sentinel
PostgreSQL (profiles-db)
auth-service
4. Golden rules
1) Check recent deployments first: roughly one in four incidents in the log is deploy related, and a rollback
is almost always faster t

In [138]:
#Pass retrieved chunks as context to LLM
query = input("Enter your query: ")

retrieved_docs = vector_store.similarity_search(query, k=5)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

prompt = f"""
Answer the question using ONLY the provided context.

Context:
{context}

Question:
{query}

Answer:
"""


Enter your query: explain application and gateway errors?


In [139]:
#Prompt Design to Ensure Grounded Responses


prompt = f"""
You are a helpful assistant answering questions based only on the provided context.

Instructions:
1. Use only the information in the context.
2. Do not use external knowledge.
3. If the answer is not available in the context, respond:
   "The provided context does not contain sufficient information to answer this question."
4. Cite relevant information from the context when possible.
5. Keep the answer concise and factual.

Context:
{context}

Question:
{query}

Answer:
"""

In [140]:
#Generate Answers Using the LLM

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

response = llm.invoke(prompt)

print(response.text)


Based on the provided runbook, application and gateway errors include:

*   **ERR-500 (Unhandled exception):** Characterized by HTTP 500 errors with stack traces. It is the most common deploy-related incident (45%), primarily affecting `order-api`, `inventory-service`, and `user-profile-api`. Causes include missing payload fields, unhandled exceptions, feature flag issues, or library upgrades.
*   **ERR-503 (Downstream dependency unavailable):** Characterized by HTTP 503 errors with "upstream connect error" or open circuit breakers. It typically affects `payment-gateway`, `notification-service`, and `search-service`.
*   **ERR-504 (Gateway timeout):** Characterized by HTTP 504 errors after 30 seconds and p95 latency exceeding 3,000 ms. Causes include slow database queries, thread pool saturation, missing timeouts on outbound calls, or under-provisioned pods.

**General diagnostic and fix strategies include:**
*   **ERR-500:** Match timestamps to deploys, group stack traces to find the 

**Embedding & Retrieval**

Document chunks embedded using gemini-embedding-2
FAISS used for vector storage and similarity search
Top-k retrieval tested with k = 3, 5, and 7

**Prompt Design**

Explicit instruction to use only retrieved context
Fallback response for missing information

**Sample Query & Answer**

Query: what should do for the standard first five minitues?

Answer: For the standard first five minutes, you should:

1. Acknowledge the alert and open an incident ticket (INC-xxxx).
2. Identify the error code and service from the alert payload.
3. Check the deploy dashboard to see if anything was released in the last 2 hours.
4. Check dependency health by reviewing the status page for each critical dependency.
5. Post the first update in #incidents with the service, code, severity, and owner.

**Evidence of Grounding**

Every answer is generated from retrieved chunks.
Answers are rejected when supporting information is absent.
No external knowledge is allowed in the prompt for minimizing hallucinations.